# E004 — CTR model comparison on the test days

**Hypotheses:** H1, H9 ([docs/hypotheses.md](../../docs/hypotheses.md))

**Question:** Does a cross network beat additive and tree baselines on the same inputs, and does validation-day calibration transfer to the test days?

**Method:** Train logistic, tuned LightGBM and a 3-seed DCN-v2 ensemble on 10-21..10-27 with early stopping on 10-28; choose identity/Platt/isotonic calibration by hour-fold CV on 10-28; score the test days once; compare with a paired hour-block bootstrap. Writes the serving bundle and the mlcheck evidence.

In [1]:
from charade.analysis.experiment import setup

ctx = setup()
f"mode: {'smoke (fixture, tiny model)' if ctx.smoke else 'full data'}"

'mode: full data'

In [2]:
import numpy as np
import polars as pl

from charade.models import pipeline

metrics = pipeline.run(ctx.settings, reports=ctx.reports / "models")
pl.DataFrame(
    [{"model": m, **{k: round(v, 4) for k, v in s["test"].items() if k != "n"}} for m, s in metrics["models"].items()]
)

W1001 14:10:45.279000 858947 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms


W1001 14:10:45.280000 858947 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::roi_align


W1001 14:10:45.281000 858947 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::roi_pool


W1001 14:10:45.281000 858947 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::deform_conv2d


/home/lucas/Developer/personal/take_home_assigment/.venv/lib/python3.12/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:486: UserWarning: # The axis name: n will not be used, since it shares the same shape constraints with another axis: n.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


model,ctr,ne,logloss,auc,calibration_ratio,ece
str,f64,f64,f64,f64,f64,f64
"""dcn_v2""",0.1714,0.8848,0.4053,0.7373,1.0121,0.0048
"""logreg""",0.1714,0.911,0.4174,0.7092,0.9849,0.0076
"""lightgbm""",0.1714,0.8934,0.4093,0.7293,0.9795,0.0102


In [3]:
pl.DataFrame(
    [{"comparison": k, **{a: round(b, 5) for a, b in v.items()}} for k, v in metrics["comparisons_test"].items()]
)

comparison,delta_logloss,ci_low,ci_high
str,f64,f64,f64
"""dcn_v2-logreg""",-0.01204,-0.01384,-0.01019
"""dcn_v2-lightgbm""",-0.00397,-0.00517,-0.00273


In [4]:
{m: c["kind"] for m, c in metrics["calibration_choice"].items()}

{'dcn_v2': 'isotonic', 'logreg': 'identity', 'lightgbm': 'platt'}

## Calibration on the test days (equal-mass bins, shipped model)

In [5]:
import matplotlib.pyplot as plt

preds = pl.read_parquet(ctx.settings.artifacts_dir / "predictions.parquet").filter(
    (pl.col("model") == "dcn_v2") & (pl.col("split") == "test")
)
order = np.argsort(preds["pred"].to_numpy())
bins = np.array_split(order, 15)
p, y = preds["pred"].to_numpy(), preds["label"].to_numpy()
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0, 0.6], [0, 0.6], color="grey", lw=1)
ax.plot([p[b].mean() for b in bins], [y[b].mean() for b in bins], "o-")
ax.set(xlabel="predicted CTR", ylabel="observed CTR", title="Reliability, test days")
plt.show()

/tmp/ipykernel_858947/1456950024.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
pl.DataFrame(metrics["slices_test"]).filter(pl.col("slice").is_in(["character_support", "day", "user_seen"])).select(
    "slice", "value", "n", pl.col("ne").round(4), pl.col("calibration_ratio").round(3)
)

slice,value,n,ne,calibration_ratio
str,str,f64,f64,f64
"""character_support""","""0 (cold)""",1478.0,0.9061,1.079
"""character_support""","""1-20""",2549.0,0.887,1.038
"""character_support""","""21-200""",30308.0,0.884,1.01
"""character_support""",""">200""",93071.0,0.8846,1.011
"""day""","""2014-10-29""",104450.0,0.8848,1.01
"""day""","""2014-10-30""",22956.0,0.8851,1.022
"""user_seen""","""new""",97843.0,0.8936,1.004
"""user_seen""","""seen""",29563.0,0.8564,1.043


## Reading
- H1: the cross network beats both baselines with CIs that exclude zero.
- H9: the per-day predicted/observed ratio stays within [0.9, 1.1] on both test days.

## Conclusion
DCN-v2 (3-seed ensemble, isotonic) reaches test NE 0.8848 and beats LightGBM by -0.0040 [-0.0052, -0.0027] and logistic by -0.0120 [-0.0138, -0.0102] log loss; calibration holds on both test days.

**Decision:** Ship the DCN-v2 ensemble; keep LightGBM and logistic as yardsticks in every run.